# Hands-on Clustering Analysis with Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leeexiaoming/web/blob/master/teaching/kmeans_clustering_iris_demo.ipynb)

## Learning objectives

By the end of this demonstration, you will be able to:

- select appropriate variables for clustering;
- prepare numerical data for K-means;
- run and interpret a K-means model;
- use WSSE and an elbow plot to compare values of K; and
- visualize and evaluate the resulting clusters.

## 1. Define the business problem

**Goal:** Identify different types of iris flowers using measurements of their sepals and petals.

K-means is an **unsupervised learning** method. It groups observations that are similar to one another without using known class labels.

## 2. Import the Python libraries

In [ ]:
import io

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

## 3. Load the data

Choose one option below:

- Keep `UPLOAD_FILE = False` to load the CSV from `raw_url`.
- Change it to `True` to display a file-upload button in Google Colab.

In [ ]:
# Option 1: Load the data directly from GitHub.
raw_url = "https://raw.githubusercontent.com/leeexiaoming/web/master/teaching/iris_csv.csv"
UPLOAD_FILE = False  # Change to True to upload a CSV in Colab

# Option 2: Display Colab's upload button when UPLOAD_FILE is True.
if UPLOAD_FILE:
    from google.colab import files

    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
else:
    df = pd.read_csv(raw_url)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 4. Understand the variables

The dataset contains six columns:

- `RowNames`: observation ID
- `SPECIES`: known flower species
- `SEPAL.L`: sepal length in centimeters
- `SEPAL.W`: sepal width in centimeters
- `PETAL.L`: petal length in centimeters
- `PETAL.W`: petal width in centimeters

In [ ]:
df.describe(include="all").round(2)

## 5. Select variables for clustering

We use the four flower measurements. We do **not** use:

- `RowNames`, because an ID does not describe the flower; or
- `SPECIES`, because K-means should discover groups without knowing the true labels.

In [ ]:
# Exclude the ID and known species label from the clustering inputs.
feature_columns = ["SEPAL.L", "SEPAL.W", "PETAL.L", "PETAL.W"]
X = df[feature_columns]
X.head()

## 6. Prepare the data

K-means cannot directly handle missing values and is sensitive to variable scales. We first check for missing data, then scale every feature to the range 0–1.

This range scaling approximates the normalized-distance behavior used by Weka.

In [ ]:
print(X.isna().sum())  # Check missing values

# Scale every measurement to the same 0–1 range.
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

## 7. Run K-means with three clusters

We begin with `K = 3`, following the case in the slides. Setting `random_state` makes the demonstration reproducible.

In [ ]:
# Fit a reproducible three-cluster model using the scaled measurements.
kmeans = KMeans(n_clusters=3, random_state=10, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

## 8. Interpret the clustering output

For each cluster, we report:

- the number of observations;
- the mean of each original measurement; and
- the within-cluster sum of squared errors (WSSE).

A lower WSSE means observations are closer to their assigned cluster centers. Here, WSSE is calculated in the scaled feature space.

In [ ]:
# Describe each cluster using counts and means in the original units.
cluster_summary = df.groupby("Cluster")[feature_columns].mean().round(2)
cluster_summary.insert(0, "Count", df.groupby("Cluster").size())

display(cluster_summary)
print(f"WSSE (scaled features): {kmeans.inertia_:.4f}")

## 9. Decide the number of clusters

The elbow method compares WSSE across several values of K. Look for the point after which adding another cluster produces only a small improvement.

In [ ]:
k_values = range(2, 9)
wsse_values = [
    KMeans(n_clusters=k, random_state=10, n_init=10).fit(X_scaled).inertia_
    for k in k_values
]

plt.figure(figsize=(7, 4))
plt.plot(k_values, wsse_values, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("WSSE")
plt.title("Elbow Plot for K-means")
plt.xticks(list(k_values))
plt.show()

## 10. Visualize the clusters

The original data have four dimensions. For a simple two-dimensional view, we plot petal length against petal width and mark the cluster centers with black X symbols.

In [ ]:
centers = scaler.inverse_transform(kmeans.cluster_centers_)

# Plot petal length and width; X marks show the cluster centers.
plt.figure(figsize=(8, 5))
plt.scatter(df["PETAL.L"], df["PETAL.W"], c=df["Cluster"], cmap="viridis")
plt.scatter(
    centers[:, 2], centers[:, 3],
    color="red",
    marker="X",
    s=200,
)
plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("K-means Clusters of Iris Flowers")
plt.colorbar(label="Cluster")
plt.show()

## 11. Compare clusters with the known species

The `SPECIES` column was **not** used to create the clusters. We use it only after clustering to understand how the discovered groups correspond to the known flower types.

Cluster numbers are arbitrary. For example, Python's Cluster 0 does not need to represent the same group as Weka's Cluster 0.

In [ ]:
# SPECIES is used only to evaluate the clusters after modeling.
pd.crosstab(df["Cluster"], df["SPECIES"], margins=True)

## 12. General clustering process

1. **Decide which variables to use.** Exclude IDs, labels, and variables that do not address the business problem.
2. **Prepare the data.** Check for missing values, outliers, skewed variables, text fields, and different scales.
3. **Run the clustering analysis.** Compare possible values of K and fit the selected model.
4. **Explain the clusters.** Use cluster sizes, variable means, WSSE, and visualizations to describe the groups.

### Key takeaway

Clustering is useful only when the groups can be connected back to a meaningful business or research question.